# Array Response Reference / Eager / Compiled Benchmarks

This notebook compares three core implementations:

- `_arrayResponseCoreReference(...)`
- `arrayResponseCore(...)`
- `arrayResponseCoreCompiled(...)`

It also compares eager vs compiled end-to-end `evaluateBatch(...)` performance across the same chunk-size and reduction-tile-cap sweep.

In [2]:
from __future__ import annotations

import importlib
import os
import statistics
import sys
import time
from pathlib import Path

import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.arraySimulation as array_simulation
import train.objective as objective_module

array_simulation = importlib.reload(array_simulation)
objective_module = importlib.reload(objective_module)

_arrayResponseCoreReference = array_simulation._arrayResponseCoreReference
arrayResponseCore = array_simulation.arrayResponseCore
arrayResponseCoreCompiled = array_simulation.arrayResponseCoreCompiled
isTorchCompileAvailable = array_simulation.isTorchCompileAvailable
resetArrayResponseCoreCompiledCache = array_simulation.resetArrayResponseCoreCompiledCache
evaluateBatch = objective_module.evaluateBatch

from train.config import loadRunConfig, resolveDevice, resolveTarget, runConfigToDict
from train.evolve import EvolutionController

print(f'project_root: {PROJECT_ROOT}')
print(f'cwd: {Path.cwd()}')
print(f'torch: {torch.__version__}')
print(f'torch.compile available: {isTorchCompileAvailable()}')

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


project_root: /home/aryan/projects/Helios
cwd: /home/aryan/projects/Helios
torch: 2.11.0+cu130
torch.compile available: True


In [ ]:
# Experiment knobs
import os
os.environ["TORCH_LOGS"] = "recompiles,graph_breaks"

CONFIG_PATH = PROJECT_ROOT / Path('configs/evo.yaml')
SWEEP = 'both'  # 'linear', 'wide', or 'both'
CHUNK_SIZES = [1_600_000]
REDUCTION_TILE_CAPS = [4096]
RUNS = 3
WARMUP = 1
ALLOW_SHARED_TARGET_FAST_PATH = True
DTYPE_OVERRIDE = None  # e.g. torch.float16 or torch.float32

# Direct-core benchmark knobs
DIRECT_CORE_GRID_SIZE = 128
DIRECT_CORE_NORMALIZE = False
DIRECT_CORE_DB = False

# torch.compile knobs
TORCH_COMPILE_BACKEND = 'inductor'
TORCH_COMPILE_MODE = 'default'  # e.g. 'reduce-overhead' or 'max-autotune'
TORCH_COMPILE_DYNAMIC = False


In [4]:
def _mean(values: list[float]) -> float:
    return statistics.mean(values) if values else 0.0


def _stdev(values: list[float]) -> float:
    return statistics.pstdev(values) if len(values) > 1 else 0.0


def _sync(device: torch.device) -> None:
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def _candidate_chunk_sizes(chunk_size: int, sweep_mode: str, evolution_config):
    if sweep_mode == 'linear':
        return chunk_size, evolution_config.wideResponseChunkSize
    if sweep_mode == 'wide':
        return evolution_config.linearResponseChunkSize, chunk_size
    return chunk_size, chunk_size


def build_controller_and_batch(config_path: Path, dtype_override=None):
    config_path = Path(config_path).resolve()
    run_config = loadRunConfig(config_path)
    target = resolveTarget(run_config)
    device, dtype = resolveDevice(run_config)
    if dtype_override is not None:
        dtype = dtype_override

    controller = EvolutionController(
        config=run_config.evolution,
        targetSpec=target,
        arraySpec=run_config.array,
        lossParams=run_config.loss,
        experimentName=run_config.experiment.name,
        archiveRoot=run_config.experiment.archiveDir,
        loggingConfig=run_config.logging,
        checkpointConfig=run_config.checkpoint,
        workerConfig=run_config.workers,
        targetMode=run_config.experiment.targetMode,
        sourceConfigPath=run_config.sourcePath,
        resolvedConfig=runConfigToDict(run_config),
        writerLogDir=run_config.experiment.logDir,
    )

    with torch.no_grad():
        batch = controller.initEvolution(dtype=dtype, device=device)
        loss_params = controller._lossParamsForStep(0)

    return run_config, controller, batch, loss_params, device, dtype


def build_direct_core_inputs(batch, grid_size: int):
    az_axis = torch.linspace(-torch.pi, torch.pi, grid_size, device=batch.device, dtype=batch.dtype)
    el_axis = torch.linspace(-torch.pi / 2, torch.pi / 2, grid_size, device=batch.device, dtype=batch.dtype)
    az_grid, el_grid = torch.meshgrid(az_axis, el_axis, indexing='ij')
    return {
        'elementLocalPosition': batch.elementLocalPosition,
        'weights': batch.weights,
        'wavelength': batch.wavelength,
        'azimuth': az_grid,
        'elevation': el_grid,
        'gain': batch.gain,
    }


def _time_call(fn, *, device: torch.device, runs: int, warmup: int):
    timings_ms = []
    output = None
    for iteration in range(warmup + runs):
        _sync(device)
        start = time.perf_counter()
        output = fn()
        _sync(device)
        elapsed_ms = (time.perf_counter() - start) * 1000.0
        if iteration >= warmup:
            timings_ms.append(elapsed_ms)
    return timings_ms, output


def benchmark_core_variants(
    *,
    batch,
    chunk_sizes: list[int],
    reduction_tile_caps: list[int],
    grid_size: int,
    runs: int,
    warmup: int,
    normalize: bool,
    dB: bool,
    compile_backend: str,
    compile_mode: str | None,
    compile_dynamic: bool,
):
    base_inputs = build_direct_core_inputs(batch, grid_size)
    rows = []
    compile_available = isTorchCompileAvailable()
    resetArrayResponseCoreCompiledCache()

    with torch.no_grad():
        for reduction_tile_cap in reduction_tile_caps:
            for chunk_size in chunk_sizes:
                kwargs = {
                    **base_inputs,
                    'chunkSize': chunk_size,
                    'reductionTileCap': reduction_tile_cap,
                    'normalize': normalize,
                    'dB': dB,
                }

                reference_timings, reference_output = _time_call(
                    lambda: _arrayResponseCoreReference(**kwargs),
                    device=batch.device,
                    runs=runs,
                    warmup=warmup,
                )
                eager_timings, eager_output = _time_call(
                    lambda: arrayResponseCore(**kwargs),
                    device=batch.device,
                    runs=runs,
                    warmup=warmup,
                )

                compiled_timings = []
                compiled_output = None
                if compile_available:
                    compiled_timings, compiled_output = _time_call(
                        lambda: arrayResponseCoreCompiled(
                            **kwargs,
                            backend=compile_backend,
                            mode=compile_mode,
                            dynamic=compile_dynamic,
                        ),
                        device=batch.device,
                        runs=runs,
                        warmup=warmup,
                    )

                eager_max_abs_delta = float((eager_output - reference_output).abs().max().item())
                compiled_max_abs_delta = (
                    float((compiled_output - reference_output).abs().max().item())
                    if compiled_output is not None
                    else None
                )

                reference_mean = _mean(reference_timings)
                eager_mean = _mean(eager_timings)
                compiled_mean = _mean(compiled_timings) if compiled_timings else None

                rows.append(
                    {
                        'requested_chunk_size': chunk_size,
                        'reduction_tile_cap': reduction_tile_cap,
                        'reference_mean_ms': reference_mean,
                        'reference_std_ms': _stdev(reference_timings),
                        'eager_mean_ms': eager_mean,
                        'eager_std_ms': _stdev(eager_timings),
                        'compiled_mean_ms': compiled_mean,
                        'compiled_std_ms': _stdev(compiled_timings) if compiled_timings else None,
                        'eager_speedup_vs_reference': reference_mean / eager_mean if eager_mean > 0 else float('nan'),
                        'compiled_speedup_vs_reference': (
                            reference_mean / compiled_mean if compiled_mean and compiled_mean > 0 else None
                        ),
                        'compiled_speedup_vs_eager': (
                            eager_mean / compiled_mean if compiled_mean and compiled_mean > 0 else None
                        ),
                        'eager_max_abs_delta': eager_max_abs_delta,
                        'compiled_max_abs_delta': compiled_max_abs_delta,
                    }
                )

    rows.sort(key=lambda row: (row['reduction_tile_cap'], row['requested_chunk_size']))
    return rows


def benchmark_evaluation_variants(
    *,
    controller,
    batch,
    loss_params,
    sweep: str,
    chunk_sizes: list[int],
    reduction_tile_caps: list[int],
    runs: int,
    warmup: int,
    allow_shared_target_fast_path: bool,
):
    rows = []
    variants = [('eager', False)]
    if isTorchCompileAvailable():
        variants.append(('compiled', True))

    with torch.no_grad():
        for variant_name, use_compiled in variants:
            if use_compiled:
                resetArrayResponseCoreCompiledCache()
            for reduction_tile_cap in reduction_tile_caps:
                for chunk_size in chunk_sizes:
                    linear_chunk_size, wide_chunk_size = _candidate_chunk_sizes(
                        chunk_size,
                        sweep,
                        controller.config,
                    )

                    timings_ms = []
                    gpu_peaks_mb = []
                    score_means = []
                    resolved_linear_chunk_size = None
                    resolved_wide_chunk_size = None

                    for iteration in range(warmup + runs):
                        _sync(batch.device)
                        start = time.perf_counter()
                        evaluation = evaluateBatch(
                            batch=batch,
                            target=controller.targetSpec,
                            params=loss_params,
                            targetMode=controller.targetMode,
                            linearResponseChunkSize=linear_chunk_size,
                            wideResponseChunkSize=wide_chunk_size,
                            responseReductionTileCap=reduction_tile_cap,
                            useCompiledArrayResponse=use_compiled,
                            allowSharedTargetFastPath=allow_shared_target_fast_path,
                        )
                        _sync(batch.device)
                        elapsed_ms = (time.perf_counter() - start) * 1000.0

                        if iteration >= warmup:
                            timings_ms.append(elapsed_ms)
                            score_means.append(float(evaluation.totalLoss.mean().item()))
                            if evaluation.diagnostics.gpuMaxMemoryMB is not None:
                                gpu_peaks_mb.append(float(evaluation.diagnostics.gpuMaxMemoryMB))

                        resolved_linear_chunk_size = evaluation.diagnostics.linearResponseChunkSize
                        resolved_wide_chunk_size = evaluation.diagnostics.wideResponseChunkSize

                    rows.append(
                        {
                            'variant': variant_name,
                            'requested_chunk_size': chunk_size,
                            'reduction_tile_cap': reduction_tile_cap,
                            'resolved_linear_chunk_size': resolved_linear_chunk_size,
                            'resolved_wide_chunk_size': resolved_wide_chunk_size,
                            'mean_total_ms': _mean(timings_ms),
                            'std_total_ms': _stdev(timings_ms),
                            'min_total_ms': min(timings_ms) if timings_ms else 0.0,
                            'max_total_ms': max(timings_ms) if timings_ms else 0.0,
                            'mean_gpu_mb': _mean(gpu_peaks_mb),
                            'mean_score': _mean(score_means),
                        }
                    )

    rows.sort(key=lambda row: (row['variant'], row['reduction_tile_cap'], row['requested_chunk_size']))
    return rows


def display_rows(rows):
    if pd is not None:
        frame = pd.DataFrame(rows)
        display(frame)
        return frame
    for row in rows:
        print(row)
    return rows


In [5]:
run_config, controller, batch, loss_params, device, dtype = build_controller_and_batch(
    CONFIG_PATH,
    dtype_override=DTYPE_OVERRIDE,
)

print(f'config: {CONFIG_PATH}')
print(f'device: {device} dtype={dtype}')
print(f'batch_size: {batch.batchSize} elements: {batch.N}')
print(f'chunk_sizes: {CHUNK_SIZES}')
print(f'reduction_tile_caps: {REDUCTION_TILE_CAPS}')
print(f'compile backend: {TORCH_COMPILE_BACKEND}')
print(f'compile mode: {TORCH_COMPILE_MODE}')
print(f'compile dynamic: {TORCH_COMPILE_DYNAMIC}')


KeyboardInterrupt: 

In [ ]:
# Direct core benchmark: reference vs eager vs compiled
core_variant_rows = benchmark_core_variants(
    batch=batch,
    chunk_sizes=CHUNK_SIZES,
    reduction_tile_caps=REDUCTION_TILE_CAPS,
    grid_size=DIRECT_CORE_GRID_SIZE,
    runs=RUNS,
    warmup=WARMUP,
    normalize=DIRECT_CORE_NORMALIZE,
    dB=DIRECT_CORE_DB,
    compile_backend=TORCH_COMPILE_BACKEND,
    compile_mode=TORCH_COMPILE_MODE,
    compile_dynamic=TORCH_COMPILE_DYNAMIC,
)

core_variant_results = display_rows(core_variant_rows)


/home/aryan/miniconda3/envs/helios/lib/python3.12/site-packages/torch/_inductor/lowering.py:2212: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
/home/aryan/miniconda3/envs/helios/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W0404 01:30:43.862000 275431 site-packages/torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


In [ ]:
# End-to-end evaluation sweep: eager vs compiled
evaluation_variant_rows = benchmark_evaluation_variants(
    controller=controller,
    batch=batch,
    loss_params=loss_params,
    sweep=SWEEP,
    chunk_sizes=CHUNK_SIZES,
    reduction_tile_caps=REDUCTION_TILE_CAPS,
    runs=RUNS,
    warmup=WARMUP,
    allow_shared_target_fast_path=ALLOW_SHARED_TARGET_FAST_PATH,
)

evaluation_variant_results = display_rows(evaluation_variant_rows)


NameError: name 'benchmark_evaluation_variants' is not defined

In [ ]:
# Optional plots
if plt is None:
    print('matplotlib is not available in this environment')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    compiled_available = any(row['compiled_mean_ms'] is not None for row in core_variant_rows)
    if compiled_available:
        for reduction_tile_cap in REDUCTION_TILE_CAPS:
            core_subset = [
                row for row in core_variant_rows if row['reduction_tile_cap'] == reduction_tile_cap
            ]
            core_subset = sorted(core_subset, key=lambda row: row['requested_chunk_size'])
            axes[0].plot(
                [row['requested_chunk_size'] for row in core_subset],
                [row['compiled_speedup_vs_eager'] for row in core_subset],
                marker='o',
                label=f'cap={reduction_tile_cap}',
            )
        axes[0].axhline(1.0, color='black', linewidth=1, linestyle='--')
        axes[0].set_title('Compiled / Eager Core Speedup')
        axes[0].set_ylabel('Eager Time / Compiled Time')
        axes[0].legend()
    else:
        axes[0].text(0.5, 0.5, 'torch.compile unavailable', ha='center', va='center')
        axes[0].set_title('Compiled / Eager Core Speedup')

    axes[0].set_xscale('log')
    axes[0].set_xlabel('Requested Chunk Size')

    for reduction_tile_cap in REDUCTION_TILE_CAPS:
        for variant_name, line_style in [('eager', '--'), ('compiled', '-')]:
            evaluation_subset = [
                row
                for row in evaluation_variant_rows
                if row['reduction_tile_cap'] == reduction_tile_cap and row['variant'] == variant_name
            ]
            if not evaluation_subset:
                continue
            evaluation_subset = sorted(evaluation_subset, key=lambda row: row['requested_chunk_size'])
            axes[1].plot(
                [row['requested_chunk_size'] for row in evaluation_subset],
                [row['mean_total_ms'] for row in evaluation_subset],
                marker='o',
                linestyle=line_style,
                label=f'{variant_name} cap={reduction_tile_cap}',
            )

    axes[1].set_xscale('log')
    axes[1].set_title('End-to-End Evaluation Time')
    axes[1].set_xlabel('Requested Chunk Size')
    axes[1].set_ylabel('Mean Total Time (ms)')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
